# Data exploration

Exploration of data structure and edge cases

## Config

In [17]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path("..") / "data"
EXPORT_DATE = pd.Timestamp("2026-10-01")

## Helper functions

In [20]:
def load_raw_csv(path: Path, encoding: str, separator: str) -> pd.DataFrame:
    """Read a CSV as raw strings, keeping blanks as empty strings.

    Args:
        path: Path to the CSV file.
        encoding: File encoding, detected from the raw bytes.
        separator: Field separator.

    Returns:
        A DataFrame where every column has string dtype and no value is
        turned into NaN.
    """
    return pd.read_csv(
        path,
        dtype=str,
        keep_default_na=False,
        encoding=encoding,
        sep=separator,
    )

In [21]:
def value_shapes(values: pd.Series) -> pd.Series:
    """Summarise string formats by masking their characters.

    Digits become 9 and letters become a or A, so distinct formats (for
    example "9999-99-99" vs "9. 9. 9999") collapse into countable patterns.

    Args:
        values: A string column.

    Returns:
        Counts of each masked pattern, most frequent first.
    """
    masked = (
        values.str.replace(r"\d", "9", regex=True)
        .str.replace(r"[a-záčďéěíňóřšťúůýž]", "a", regex=True)
        .str.replace(r"[A-ZÁČĎÉĚÍŇÓŘŠŤÚŮÝŽ]", "A", regex=True)
    )
    return masked.value_counts()

In [22]:
def profile_columns(frame: pd.DataFrame) -> pd.DataFrame:
    """Describe each column by blanks, distinct values and a first example.

    Args:
        frame: Any raw string DataFrame.

    Returns:
        One row per column with the count of empty strings, the number of
        unique values and the first non-empty value seen.
    """
    return pd.DataFrame(
        {
            "empty": (frame == "").sum(),
            "unique": frame.nunique(),
            "example": frame.apply(lambda col: col[col != ""].iloc[0] if (col != "").any() else ""),
        }
    )

## 1. Loading

Encodings and separators were detected from the raw bytes

In [24]:
employees_df = load_raw_csv(DATA_DIR / "zamestnanci.csv", encoding="utf-8-sig", separator=",")
completed_df = load_raw_csv(DATA_DIR / "absolvovana_skoleni.csv", encoding="cp1250", separator=";")
catalog_df = load_raw_csv(DATA_DIR / "katalog_skoleni.csv", encoding="utf-8-sig", separator=",")
dates_df = load_raw_csv(DATA_DIR / "terminy_skoleni.csv", encoding="utf-8-sig", separator=";")

In [26]:
raw_frames = {
    "employees": employees_df,
    "completed": completed_df,
    "catalog": catalog_df,
    "dates": dates_df,
}
for name, frame in raw_frames.items():
    rows, columns = frame.shape
    print(f"{name:<10}{rows:>5} rows {columns:>3} columns")
    print(f"{'':<10}{', '.join(frame.columns)}")
    print()

employees   174 rows  12 columns
          id_zamestnance, jmeno, prijmeni, email, utvar, pozice, vedouci_email, datum_nastupu, datum_ukonceni, stav, rodne_cislo, poznamka

completed  2868 rows   9 columns
          id_zaznamu, id_zamestnance, zamestnanec, kod_skoleni, datum_absolvovani, platnost_do, poskytovatel, cena_kc, poznamka

catalog      10 rows   5 columns
          kod, nazev, platnost_mesicu, povinne_pro, poznamka

dates        34 rows   9 columns
          id_terminu, kod_skoleni, datum, cas, misto, kapacita, obsazeno, cena_na_osobu, stav



## 2. Catalog (`katalog_skoleni.csv`)


In [27]:
catalog_df

,kod,nazev,platnost_mesicu,povinne_pro,poznamka
0,BOZP-Z,BOZP – základní školení,24,VSICHNI,
1,POZ-OCH,Požární ochrana,24,VSICHNI,
2,GDPR,Ochrana osobních údajů,36,VSICHNI,
3,KYBER,Kybernetická bezpečnost,12,VSICHNI,
4,PRV-POM,První pomoc,36,UTVAR:Laboratoře;Správa budov;Knihovna,
5,RIDIC,Řidiči referenti,12,POZICE:Řidič,
6,LAB-BEZ,Bezpečná práce v laboratoři,24,UTVAR:Laboratoře,
7,ZDVIH,Obsluha zdvihacích zařízení,2 roky,POZICE:Technik údržby,
8,LEAD,Základy vedení lidí,0,POZICE:Vedoucí,
9,ANGL,Anglický jazyk – kurz,,NEPOVINNE,


## 3. Employees (`zamestnanci.csv`)

`rodne_cislo` - personal data, therefore dropped immediately after loading.

In [28]:
print(f"rodne_cislo non-empty: {(employees_df['rodne_cislo'] != '').sum()} of {len(employees_df)}")


rodne_cislo non-empty: 174 of 174


,id_zamestnance,jmeno,prijmeni,email,utvar,pozice,vedouci_email,datum_nastupu,datum_ukonceni,stav,poznamka
0,E0001,Tereza,Urbanová,tereza.urbanova@vzorova.example,Studijní oddělení,Vedoucí,jitka.simkova@vzorova.example,2008-02-23,,aktivní,
1,E0002,Tereza,Jandová,tereza.jandova @vzorova.example,Studijní oddělení,Specialista,tereza.urbanova@vzorova.example,2013-01-05,,aktivní,
2,E0003,Anna,Fialová,anna.fialova@vzorova.example,Studijní,Specialista,tereza.urbanova@vzorova.example,2012-03-09,,aktivní,
3,E0004,Monika,Hrušková,monika.hruskova@vzorova.example,Studijní oddělení,Specialista,tereza.urbanova@vzorova.example,2022-08-20,,aktivní,
4,E0005,Iveta,Konečná,iveta.konecna@vzorova.example,Studijní oddělení,Asistent,tereza.urbanova@vzorova.example,2007-07-03,,aktivní,
5,E0006,Klára,Blažková,klara.blazkova@vzorova.example,Studijní oddělení,Referent,tereza.urbanova@vzorova.example,2023-02-06,,aktivní,
6,E0007,Libor,Tomášek,libor.tomasek@vzorova.example,,Asistent,tereza.urbanova@vzorova.example,2015-03-01,,aktivní,
7,E0008,Kateřina,Kopecká,katerina.kopecka@vzorova.example,Studijní oddělení,Asistent,tereza.urbanova@vzorova.example,2007-04-24,,aktivní,
8,E0009,Pavla,Hájková,pavla.hajkova@vzorova.example,Studijní oddělení,Asistent,tereza.urbanova@vzorova.example,2016-12-26,2027-01-20,aktivní,
9,E0010,Daniel,Štěpánek,daniel.stepanek@vzorova.example,Studijní oddělení,Referent,tereza.urbanova@vzorova.example,2010-05-10,,aktivní,


In [29]:
profile_columns(employees_df)

,empty,unique,example
id_zamestnance,0,174,E0001
jmeno,0,65,Tereza
prijmeni,0,103,Urbanová
email,0,172,tereza.urbanova@vzorova.example
utvar,5,18,Studijní oddělení
pozice,0,11,Vedoucí
vedouci_email,1,11,jitka.simkova@vzorova.example
datum_nastupu,0,166,2008-02-23
datum_ukonceni,158,17,2027-01-20
stav,0,3,aktivní


Detecting unexpected or inconsistent category values: unknown departments or positions, casing and spelling variants, and whether `stav` is drawn from a small fixed set.

In [31]:
for column in ["stav", "utvar", "pozice"]:
    print(employees_df[column].value_counts(dropna=False))
    print()

stav
aktivní       156
ukončen        12
rodičovská      6
Name: count, dtype: int64

utvar
Laboratoře             31
Správa budov           27
IT oddělení            22
Knihovna               21
Studijní oddělení      20
Ekonomické oddělení    19
Personální oddělení     8
Marketing               6
                        5
Vedení                  5
Laboratore              3
Studijní                1
Studijni oddeleni       1
Sprava budov            1
SPRÁVA BUDOV            1
správa budov            1
Personalni oddeleni     1
VEDENÍ                  1
Name: count, dtype: int64

pozice
Laborant          28
Specialista       24
Referent          24
Technik           20
Technik údržby    16
Knihovník         15
Vedoucí           13
Účetní            12
Asistent          10
Řidič              8
Vývojář            4
Name: count, dtype: int64



Detecting which date formats appear in the employee dates, and whether more than one format (or an invalid value) is mixed in one column.

In [32]:
for column in ["datum_nastupu", "datum_ukonceni"]:
    print(column)
    print(value_shapes(employees_df[column]))
    print()

datum_nastupu
datum_nastupu
9999-99-99     169
9. 9. 9999       3
99. 9. 9999      2
Name: count, dtype: int64

datum_ukonceni
datum_ukonceni
              158
9999-99-99     16
Name: count, dtype: int64



Detecting duplicate employee ids or emails, and whether every id follows one consistent pattern.

In [33]:
print(f"duplicate id_zamestnance: {employees_df['id_zamestnance'].duplicated().sum()}")
print(f"duplicate email: {employees_df['email'].str.lower().duplicated().sum()}")
print("id_zamestnance patterns:")
print(value_shapes(employees_df["id_zamestnance"]))

duplicate id_zamestnance: 0
duplicate email: 2
id_zamestnance patterns:
id_zamestnance
A9999    174
Name: count, dtype: int64


Detecting end dates that disagree with status: active people that still carry an end date, or terminated people without one.

In [34]:
pd.crosstab(employees_df["stav"], employees_df["datum_ukonceni"] != "")

datum_ukonceni,False,True
stav,,
aktivní,152,4
rodičovská,6,0
ukončen,0,12


Detecting manager references that point to no known employee: orphaned or placeholder `vedouci_email` values.

In [35]:
known_emails = set(employees_df["email"].str.lower())
unknown_managers_df = employees_df[
    (employees_df["vedouci_email"] != "")
    & ~employees_df["vedouci_email"].str.lower().isin(known_emails)
]
print(f"vedouci_email not found among employees: {len(unknown_managers_df)}")
unknown_managers_df[["id_zamestnance", "vedouci_email"]].head(20)

vedouci_email not found among employees: 3


,id_zamestnance,vedouci_email
22,E0023,nezname.jmeno@vzorova.example
79,E0080,nezname.jmeno@vzorova.example
96,E0097,nezname.jmeno@vzorova.example


In [36]:
employees_df[employees_df["poznamka"] != ""][["id_zamestnance", "stav", "poznamka"]]

,id_zamestnance,stav,poznamka
65,E0066,aktivní,-2+3
68,E0069,aktivní,@SUM(1+1)
107,E0108,aktivní,+1+1
152,E0153,aktivní,"=HYPERLINK(""https://example.com/prihlaseni"";""K..."


## 4. Completed trainings (`absolvovana_skoleni.csv`)

In [37]:
completed_df.head(10)

,id_zaznamu,id_zamestnance,zamestnanec,kod_skoleni,datum_absolvovani,platnost_do,poskytovatel,cena_kc,poznamka
0,Z000001,E0131,Vacková Hana,BOZP-Z,1900-01-01,1902-01-01,Interní školitel,"0,00",
1,Z000002,E0149,Navrátil Miroslav,BOZP-Z,1. 1. 1900,1. 1. 1902,Interní školitel,"0,00",
2,Z000003,E0024,Blažek Adam,BOZP-Z,2019-01-06,2021-01-06,Interní školitel,"450,00",
3,Z000004,E0063,Nováková Klára,GDPR,6. 1. 2019,6. 1. 2022,Vzorový školicí ústav,"700,00",
4,Z000005,E0088,Matoušková Adéla,POZ-OCH,6. 1. 2019,6. 1. 2021,E-learning portál,"380,00",online verze
5,Z000006,E0096,Novák Daniel,POZ-OCH,6. 1. 2019,6. 1. 2021,Vzorový školicí ústav,"0,00",
6,Z000007,E0118,Konečný Pavel,BOZP-Z,6. 1. 2019,6. 1. 2021,Vzorový školicí ústav,"550,00",
7,Z000008,E0067,Poláková Kateřina,BOZP-Z,7. 1. 2019,7. 1. 2021,Externí lektor,"550,00",absolvováno v zahraničí
8,Z000009,E0094,Pospíšil Vladimír,GDPR,07.01.2019,07.01.2024,Vzorový školicí ústav,"550,00",
9,Z000010,E0103,Prokopová Lenka,GDPR,7. 1. 2019,7. 1. 2022,E-learning portál,"650,00",


In [38]:
profile_columns(completed_df)

,empty,unique,example
id_zaznamu,0,2853,Z000001
id_zamestnance,2,182,E0131
zamestnanec,2,174,Vacková Hana
kod_skoleni,2,31,BOZP-Z
datum_absolvovani,2,1985,1900-01-01
platnost_do,49,2057,1902-01-01
poskytovatel,2,10,Interní školitel
cena_kc,21,47,"0,00"
poznamka,2599,14,online verze


Detecting the mix of formats in the completion and validity dates (ISO, `d. m. Y`, and any values that are not dates at all).

In [39]:
for column in ["datum_absolvovani", "platnost_do"]:
    print(column)
    print(value_shapes(completed_df[column]))
    print()

datum_absolvovani
datum_absolvovani
99. 9. 9999          968
9999-99-99           705
99.99.9999           462
9. 9. 9999           397
99. 99. 9999         242
9. 99. 9999           88
99999                  3
                       2
aaaaa_aaaaaaaaaaa      1
Name: count, dtype: int64

platnost_do
platnost_do
99. 9. 9999     954
9999-99-99      692
99.99.9999      449
9. 9. 9999      390
99. 99. 9999    243
9. 99. 9999      87
                 49
99999             3
aaaaaaaa_aa       1
Name: count, dtype: int64



Detecting training codes that match no catalog code: typos, casing or spacing variants, and genuinely unknown codes.

In [41]:
print("kod_skoleni not in catalog:")
completed_df.loc[~completed_df["kod_skoleni"].isin(catalog_df["kod"]), "kod_skoleni"].value_counts()

kod_skoleni not in catalog:


kod_skoleni
KYBER          6
BOZP-X         4
 BOZP-Z        2
BOZP-Z         2
POZ_OCH        2
               2
 POZ-OCH       2
POZ-OCH        2
POZ OCH        2
lead           1
BOZP Z         1
 KYBER         1
bozp-z         1
 GDPR          1
GDPR           1
kod_skoleni    1
GDPR-2         1
kyber          1
BOZP_Z         1
LAB-BEZ        1
KYB3R          1
Name: count, dtype: int64

Detecting completions linked to an employee id that is absent from the employee table.

In [42]:
print("id_zamestnance not in employees:")
completed_df.loc[~completed_df["id_zamestnance"].isin(employees_df["id_zamestnance"]), "id_zamestnance"].value_counts()

id_zamestnance not in employees:


id_zamestnance
                  2
E9006             1
E9001             1
id_zamestnance    1
E9008             1
E9009             1
E9007             1
E9004             1
E9002             1
E9003             1
E9005             1
Name: count, dtype: int64

In [45]:
print(completed_df["kod_skoleni"].value_counts())
print()
print("poskytovatel:")
print(completed_df["poskytovatel"].value_counts())

kod_skoleni
KYBER          838
BOZP-Z         573
POZ-OCH        528
GDPR           436
PRV-POM        212
LAB-BEZ        114
RIDIC           51
ZDVIH           46
ANGL            24
LEAD            10
KYBER            6
BOZP-X           4
 BOZP-Z          2
BOZP-Z           2
POZ_OCH          2
                 2
 POZ-OCH         2
POZ-OCH          2
POZ OCH          2
lead             1
BOZP Z           1
 KYBER           1
bozp-z           1
 GDPR            1
GDPR             1
kod_skoleni      1
GDPR-2           1
kyber            1
BOZP_Z           1
LAB-BEZ          1
KYB3R            1
Name: count, dtype: int64

poskytovatel:
poskytovatel
E-learning portál        721
Vzorový školicí ústav    720
Externí lektor           713
Interní školitel         701
VZOROVÝ ŠKOLICÍ ÚSTAV      3
E-LEARNING PORTÁL          3
                           2
INTERNÍ ŠKOLITEL           2
EXTERNÍ LEKTOR             2
poskytovatel               1
Name: count, dtype: int64


Detecting the number format used for prices (decimal comma, thousands separators) so the cleaning step can parse it.

In [46]:
value_shapes(completed_df["cena_kc"])

cena_kc
999,99      1685
9,99         844
9 999,99     303
              21
999,- Ač       4
aaaaaa         4
-999,99        3
9,- Ač         2
aaaa_aa        1
9999           1
Name: count, dtype: int64

Detecting duplicate completion records at three levels of strictness: a repeated id, identical rows, and the same person + training + date.

In [47]:
print(f"duplicate id_zaznamu: {completed_df['id_zaznamu'].duplicated().sum()}")
print(f"exact duplicate rows ignoring id_zaznamu: {completed_df.drop(columns='id_zaznamu').duplicated().sum()}")
print(f"same employee + training + date: {completed_df.duplicated(['id_zamestnance', 'kod_skoleni', 'datum_absolvovani']).sum()}")

duplicate id_zaznamu: 15
exact duplicate rows ignoring id_zaznamu: 7
same employee + training + date: 12


Detecting whether the denormalised `zamestnanec` name agrees with the employee record found by id, which exposes wrong name order, dropped diacritics, or a wrong link.

In [48]:
full_names = employees_df.assign(
    full=employees_df["prijmeni"] + " " + employees_df["jmeno"],
).set_index("id_zamestnance")["full"]
name_check_df = completed_df.assign(expected=completed_df["id_zamestnance"].map(full_names))
mismatch_df = name_check_df[
    name_check_df["expected"].notna() & (name_check_df["zamestnanec"] != name_check_df["expected"])
]
print(f"{len(mismatch_df)} mismatches")
mismatch_df[["id_zamestnance", "zamestnanec", "expected"]].drop_duplicates().head(20)

2856 mismatches


,id_zamestnance,zamestnanec,expected
0,E0131,Vacková Hana,Hana Vacková
1,E0149,Navrátil Miroslav,Miroslav Navrátil
2,E0024,Blažek Adam,Adam Blažek
3,E0063,Nováková Klára,Klára Nováková
4,E0088,Matoušková Adéla,Adéla Matoušková
5,E0096,Novák Daniel,Daniel Novák
6,E0118,Konečný Pavel,Pavel Konečný
7,E0067,Poláková Kateřina,Kateřina Poláková
8,E0094,Pospíšil Vladimír,Vladimír Pospíšil
9,E0103,Prokopová Lenka,Lenka Prokopová


In [49]:
completed_df[completed_df["poznamka"] != ""][
    ["id_zaznamu", "id_zamestnance", "kod_skoleni", "datum_absolvovani", "poznamka"]
].head(30)

,id_zaznamu,id_zamestnance,kod_skoleni,datum_absolvovani,poznamka
4,Z000005,E0088,POZ-OCH,6. 1. 2019,online verze
7,Z000008,E0067,BOZP-Z,7. 1. 2019,absolvováno v zahraničí
13,Z000014,E0167,POZ-OCH,2019-01-07,bez certifikátu
49,Z000050,E0071,PRV-POM,16. 1. 2019,prezenčně
51,Z000052,E0125,GDPR,16. 1. 2019,doplněno ručně
52,Z000052,E0125,GDPR,2019-01-16,doplněno ručně
64,Z000064,E0120,BOZP-Z,17. 1. 2019,po upomínce
65,Z000065,E0130,BOZP-Z,17. 1. 2019,bez certifikátu
112,Z000112,E0022,KYBER,26. 1. 2019,prezenčně
118,Z000118,E0132,KYBER,2019-01-26,online verze


Detecting placeholder completion dates (the `1900` sentinel) that stand in for a missing real date.

In [50]:
placeholder_mask = completed_df["datum_absolvovani"].str.contains("1900")
print(f"{placeholder_mask.sum()} rows")
completed_df[placeholder_mask].head(10)

2 rows


,id_zaznamu,id_zamestnance,zamestnanec,kod_skoleni,datum_absolvovani,platnost_do,poskytovatel,cena_kc,poznamka
0,Z000001,E0131,Vacková Hana,BOZP-Z,1900-01-01,1902-01-01,Interní školitel,"0,00",
1,Z000002,E0149,Navrátil Miroslav,BOZP-Z,1. 1. 1900,1. 1. 1902,Interní školitel,"0,00",


## 5. Training dates (`terminy_skoleni.csv`)

Small enough to read in full.

In [51]:
dates_df

,id_terminu,kod_skoleni,datum,cas,misto,kapacita,obsazeno,cena_na_osobu,stav
0,T001,KYBER,2026-10-14,09:00-12:00,Aula A,25,22,800,plánováno
1,T002,KYBER,2026-10-28,09:00-12:00,Aula A,40,0,800,plánováno
2,T003,KYBER,2026-11-11,09:00-12:00,Aula A,30,5,800,plánováno
3,T004,KYBER,2026-11-25,09:00-12:00,Aula A,30,0,800,plánováno
4,T005,KYBER,2026-12-09,09:00-12:00,Aula A,25,0,800,plánováno
5,T006,KYBER,2026-12-16,13:00-15:00,Online,50,0,800,plánováno
6,T007,KYBER,2026-11-14,09:00-12:00,Online,60,0,800,plánováno
7,T008,KYBER,2026-09-23,09:00-12:00,Aula A,30,0,800,plánováno
8,T009,KYBER,2026-11-11,09:00-12:00,Aula A,30,5,800,plánováno
9,T010,KYB3R,2026-12-02,09:00-12:00,Aula B,20,0,800,plánováno


In [52]:
profile_columns(dates_df)

,empty,unique,example
id_terminu,0,34,T001
kod_skoleni,0,10,KYBER
datum,0,31,2026-10-14
cas,0,10,09:00-12:00
misto,0,9,Aula A
kapacita,0,11,25
obsazeno,0,13,22
cena_na_osobu,2,11,800
stav,0,2,plánováno


Detecting the status and location vocabularies, any term whose training code is not in the catalog, duplicate term ids, and the date format.

In [53]:
for column in ["stav", "misto", "cas"]:
    print(dates_df[column].value_counts())
    print()
print(f"kod_skoleni not in catalog: {(~dates_df['kod_skoleni'].isin(catalog_df['kod'])).sum()}")
print(f"duplicate id_terminu: {dates_df['id_terminu'].duplicated().sum()}")
print(value_shapes(dates_df["datum"]))

stav
plánováno    32
zrušeno       2
Name: count, dtype: int64

misto
Aula A                12
Online                 5
Učebna 1               4
Učebna 2               4
Autoškola Vzor         3
Laboratoř 3            2
Dvůr – sklad           2
Aula B                 1
Seminární místnost     1
Name: count, dtype: int64

cas
09:00-12:00    9
08:00-11:00    5
10:00-12:00    4
09:00-15:00    4
14:00-16:00    3
08:00-12:00    3
09:00-13:00    2
07:30-15:30    2
13:00-15:00    1
09:00-17:00    1
Name: count, dtype: int64

kod_skoleni not in catalog: 1
duplicate id_terminu: 0
datum
9999-99-99     33
99. 9. 9999     1
Name: count, dtype: int64


Detecting non-numeric values in the numeric columns, and any term booked beyond its capacity (`obsazeno > kapacita`).

In [54]:
capacity_df = dates_df[["kapacita", "obsazeno", "cena_na_osobu"]].apply(pd.to_numeric, errors="coerce")
print(f"non-numeric values: {capacity_df.isna().sum().to_dict()}")
print("obsazeno > kapacita:")
dates_df[capacity_df["obsazeno"] > capacity_df["kapacita"]]

non-numeric values: {'kapacita': 0, 'obsazeno': 0, 'cena_na_osobu': 3}
obsazeno > kapacita:


,id_terminu,kod_skoleni,datum,cas,misto,kapacita,obsazeno,cena_na_osobu,stav
15,T016,BOZP-Z,2026-11-04,08:00-11:00,Aula A,30,31,450,plánováno
29,T030,LAB-BEZ,2026-10-29,09:00-13:00,Laboratoř 3,20,21,1500,plánováno


Detecting which terms fall before or after the export date, to separate usable future terms from ones already in the past.

Note: this cell parses ISO dates only, so non-ISO or invalid dates (e.g. T014 `31. 9. 2026`) become `NaT` and show a blank distance. Full multi-format parsing is done in the cleaning notebook.

In [56]:
parsed_dates = pd.to_datetime(dates_df["datum"], format="%Y-%m-%d", errors="coerce")
dates_df.assign(
    parsed=parsed_dates,
    days_from_export=(parsed_dates - EXPORT_DATE).dt.days,
)[["id_terminu", "kod_skoleni", "datum", "stav", "days_from_export"]].sort_values("days_from_export").head(15)

,id_terminu,kod_skoleni,datum,stav,days_from_export
18,T019,BOZP-Z,2026-09-16,plánováno,-15.0
7,T008,KYBER,2026-09-23,plánováno,-8.0
21,T022,GDPR,2026-09-30,plánováno,-1.0
14,T015,BOZP-Z,2026-10-07,plánováno,6.0
0,T001,KYBER,2026-10-14,plánováno,13.0
26,T027,RIDIC,2026-10-15,plánováno,14.0
10,T011,POZ-OCH,2026-10-20,plánováno,19.0
22,T023,PRV-POM,2026-10-22,plánováno,21.0
1,T002,KYBER,2026-10-28,plánováno,27.0
29,T030,LAB-BEZ,2026-10-29,plánováno,28.0
